# 4.1 · LSTM para caudal del Genil con Keras

**Tiempo estimado:** 1 h (15 min de pipeline + 45 min de modelo).

**Objetivos.**

1. Preparar ventanas temporales `(N, T, F)` correctamente — sin fuga.
2. Normalizar por **train** (no test).
3. Construir y entrenar una LSTM con callbacks (`EarlyStopping`, `ReduceLROnPlateau`).
4. Predecir y comparar contra los baselines de Sesión 3.

## Mini-intro (15 min)

**Tensor 3D**: `(batch, timesteps, features)`. Por ejemplo, para predecir el caudal de hoy usando los 30 días anteriores y la lluvia de ayer: `T=30`, `F=2` (caudal + lluvia).

**Pipeline:**

1. Construir ventanas $(X_i, y_i)$ donde $X_i = (\text{caudal}_{t-30}, ..., \text{caudal}_{t-1}, \text{lluvia}_{t-1})$ y $y_i = \text{caudal}_t$.
2. Split temporal (no aleatorio).
3. Normalizar (fit en train, transform en test).
4. Definir LSTM en Keras `Sequential`.
5. Entrenar con `EarlyStopping`.
6. Predecir, des-normalizar, evaluar.

In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import keras
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout, Input
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler

from cst import datos as ud

plt.rcParams.update({"figure.figsize": (10, 3.4), "axes.grid": True, "grid.alpha": 0.3})
keras.utils.set_random_seed(0)

## 1 · Datos

In [ ]:
caudal = ud.cargar_caudal_genil(source="CEDEX").loc["1995":"2020"]
lluvia = ud.cargar_lluvia_genil_diaria(fecha_inicio="1995-01-01", fecha_fin="2020-12-31")
df = pd.DataFrame({"caudal": caudal, "lluvia": lluvia}).dropna(subset=["caudal"])

# Para LSTM necesitamos índice diario regular sin NaN
df = df.asfreq("D").interpolate("linear", limit=3).dropna()
print(f"Días: {len(df):,}    {df.index.min().date()} → {df.index.max().date()}")

## 2 · Ventanas (B, T, F)

In [ ]:
VENTANA = 60  # 60 días de historia
HORIZONTE = 1  # predecir un día por delante
FEATURES = ["caudal", "lluvia"]


def crear_ventanas(df, features, target, ventana, horizonte):
    X, y, idx = [], [], []
    valores = df[features].values
    target_arr = df[target].values
    fechas = df.index
    for i in range(len(valores) - ventana - horizonte + 1):
        X.append(valores[i : i + ventana])
        y.append(target_arr[i + ventana + horizonte - 1])
        idx.append(fechas[i + ventana + horizonte - 1])
    return np.asarray(X), np.asarray(y), pd.DatetimeIndex(idx)


X_all, y_all, idx_all = crear_ventanas(df, FEATURES, "caudal", VENTANA, HORIZONTE)
print(f"X shape: {X_all.shape}   y shape: {y_all.shape}")

## 3 · Split temporal y normalización

**Crítico:** el scaler se ajusta SÓLO en el train.

In [ ]:
split_test = pd.Timestamp("2018-01-01")
split_val = pd.Timestamp("2016-01-01")  # último año del train como validación

mask_tr = idx_all < split_val
mask_val = (idx_all >= split_val) & (idx_all < split_test)
mask_te = idx_all >= split_test

print(f"Train: {mask_tr.sum():,}  Val: {mask_val.sum():,}  Test: {mask_te.sum():,}")

# Reshape para escalado: (N*T, F)
N, T, F = X_all.shape
scaler_X = StandardScaler().fit(X_all[mask_tr].reshape(-1, F))
scaler_y = StandardScaler().fit(y_all[mask_tr].reshape(-1, 1))


def transform(X, y):
    Xs = scaler_X.transform(X.reshape(-1, F)).reshape(X.shape)
    ys = scaler_y.transform(y.reshape(-1, 1)).flatten()
    return Xs, ys


X_tr, y_tr = transform(X_all[mask_tr], y_all[mask_tr])
X_va, y_va = transform(X_all[mask_val], y_all[mask_val])
X_te, y_te = transform(X_all[mask_te], y_all[mask_te])
print("Normalizado.")

## 4 · Modelo LSTM

In [ ]:
def construir_lstm(ventana, n_features, units=32, dropout=0.2):
    m = Sequential(
        [
            Input(shape=(ventana, n_features)),
            LSTM(units, return_sequences=False),
            Dropout(dropout),
            Dense(1),
        ]
    )
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
    return m


model = construir_lstm(VENTANA, len(FEATURES), units=32, dropout=0.2)
model.summary()

In [ ]:
callbacks = [
    EarlyStopping(patience=8, restore_best_weights=True, monitor="val_loss"),
    ReduceLROnPlateau(patience=4, factor=0.5, monitor="val_loss"),
]

historia = model.fit(
    X_tr,
    y_tr,
    validation_data=(X_va, y_va),
    epochs=40,
    batch_size=64,
    callbacks=callbacks,
    verbose=2,
)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(historia.history["loss"], label="train")
ax.plot(historia.history["val_loss"], label="val")
ax.set_xlabel("epoch")
ax.set_ylabel("MSE (escala normalizada)")
ax.legend()
ax.set_title("Curva de entrenamiento")
plt.tight_layout()

## 5 · Predicción y des-normalización

In [ ]:
pred_norm = model.predict(X_te, verbose=0).flatten()
pred = scaler_y.inverse_transform(pred_norm.reshape(-1, 1)).flatten()
obs = y_all[mask_te]  # ya en escala original
idx_test = idx_all[mask_te]

rmse = float(np.sqrt(((obs - pred) ** 2).mean()))
mae = float(np.abs(obs - pred).mean())
print(f"LSTM h=1   RMSE={rmse:.3f}   MAE={mae:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(idx_test, obs, color="black", lw=1, label="observado")
ax.plot(idx_test, pred, color="#c2410c", lw=1, ls="--", label="LSTM")
ax.set_ylabel("Q (m³/s)")
ax.legend()
ax.set_title("LSTM (60d → 1d) — test 2018-2020")
plt.tight_layout()

## 6 · Múltiples semillas (recomendación obligatoria)

In [ ]:
preds_ensemble = []
for seed in range(3):  # 3 para velocidad en clase; en producción ≥ 5
    keras.utils.set_random_seed(seed)
    m = construir_lstm(VENTANA, len(FEATURES), units=32, dropout=0.2)
    m.fit(
        X_tr,
        y_tr,
        validation_data=(X_va, y_va),
        epochs=30,
        batch_size=64,
        callbacks=[EarlyStopping(patience=6, restore_best_weights=True)],
        verbose=0,
    )
    p = scaler_y.inverse_transform(m.predict(X_te, verbose=0)).flatten()
    preds_ensemble.append(p)
preds_ensemble = np.array(preds_ensemble)
media = preds_ensemble.mean(axis=0)
std = preds_ensemble.std(axis=0)

print(
    f"RMSE individual (3 seeds): {[np.sqrt(((obs - p) ** 2).mean()).round(3) for p in preds_ensemble]}"
)
print(f"RMSE ensemble: {np.sqrt(((obs - media) ** 2).mean()):.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(idx_test, obs, color="black", lw=1, label="observado")
ax.plot(idx_test, media, color="#c2410c", lw=1.2, label="LSTM ensemble (media)")
ax.fill_between(
    idx_test, media - std, media + std, color="#c2410c", alpha=0.2, label="±1σ entre seeds"
)
ax.set_ylabel("Q (m³/s)")
ax.legend()
plt.tight_layout()

## 7 · Multi-horizonte (MIMO)

Hasta aquí hemos predicho **un solo paso** ($y_{t+1}$). En operación real (alerta de crecidas, gestión de embalses) se quiere un **horizonte de varios días** — p. ej. $h=7$.

**Estrategia MIMO** (Multi-Input Multi-Output, *one-shot*): un único modelo cuya capa de salida es `Dense(h)` produce los $h$ valores futuros **en paralelo**. Respecto al modelo de 1 paso cambian dos cosas:

1. `crear_ventanas` devuelve `Y` con shape `(N, h)` — un vector por muestra.
2. La capa final pasa de `Dense(1)` a `Dense(h)`.

> Variante **sparse MIMO** para horizontes no consecutivos (p. ej. sólo $h \in \{7, 30\}$): mismo esquema con `Dense(len(horizontes))` y un `Y` indexado en esos pasos concretos — ver ejercicio 5.

In [ ]:
HORIZONTE_MULTI = 7  # predecir 7 días en una sola pasada


def crear_ventanas_multistep(df, features, target, ventana, horizonte):
    """Como crear_ventanas, pero Y tiene shape (N, horizonte)."""
    X, Y, idx = [], [], []
    valores = df[features].values
    target_arr = df[target].values
    fechas = df.index
    for i in range(len(valores) - ventana - horizonte + 1):
        X.append(valores[i : i + ventana])
        Y.append(target_arr[i + ventana : i + ventana + horizonte])
        idx.append(fechas[i + ventana])  # primer paso del horizonte
    return np.asarray(X), np.asarray(Y), pd.DatetimeIndex(idx)


X_m, Y_m, idx_m = crear_ventanas_multistep(df, FEATURES, "caudal", VENTANA, HORIZONTE_MULTI)
print(f"X shape: {X_m.shape}   Y shape: {Y_m.shape}")

In [ ]:
mask_tr_m = idx_m < split_val
mask_va_m = (idx_m >= split_val) & (idx_m < split_test)
mask_te_m = idx_m >= split_test


def transform_multi(X, Y):
    Xs = scaler_X.transform(X.reshape(-1, F)).reshape(X.shape)
    Ys = scaler_y.transform(Y.reshape(-1, 1)).reshape(Y.shape)
    return Xs, Ys


X_tr_m, Y_tr_m = transform_multi(X_m[mask_tr_m], Y_m[mask_tr_m])
X_va_m, Y_va_m = transform_multi(X_m[mask_va_m], Y_m[mask_va_m])
X_te_m, Y_te_m = transform_multi(X_m[mask_te_m], Y_m[mask_te_m])
print(f"Train: {len(X_tr_m):,}  Val: {len(X_va_m):,}  Test: {len(X_te_m):,}")

In [ ]:
def construir_lstm_mimo(ventana, n_features, horizonte, units=32, dropout=0.2):
    m = Sequential(
        [
            Input(shape=(ventana, n_features)),
            LSTM(units, return_sequences=False),
            Dropout(dropout),
            Dense(horizonte),  # <-- cambio MIMO
        ]
    )
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
    return m


keras.utils.set_random_seed(0)
model_mimo = construir_lstm_mimo(VENTANA, len(FEATURES), HORIZONTE_MULTI, units=32, dropout=0.2)
historia_mimo = model_mimo.fit(
    X_tr_m,
    Y_tr_m,
    validation_data=(X_va_m, Y_va_m),
    epochs=30,
    batch_size=64,
    callbacks=callbacks,
    verbose=2,
)

In [ ]:
pred_norm_m = model_mimo.predict(X_te_m, verbose=0)  # (N_te, h)
pred_m = scaler_y.inverse_transform(pred_norm_m.reshape(-1, 1)).reshape(pred_norm_m.shape)
obs_m = Y_m[mask_te_m]  # escala original

rmse_por_h = np.sqrt(((obs_m - pred_m) ** 2).mean(axis=0))
print("RMSE por paso de horizonte:")
for h_step, r in enumerate(rmse_por_h, start=1):
    print(f"  h=+{h_step}d:  RMSE={r:.3f}")
print(f"RMSE medio (h=1..{HORIZONTE_MULTI}): {rmse_por_h.mean():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
n_show = 5
sample_idx = np.linspace(0, len(pred_m) - 1, n_show).astype(int)
colors = plt.cm.viridis(np.linspace(0.15, 0.85, n_show))
issue_dates = idx_m[mask_te_m]

for i, c in zip(sample_idx, colors):
    base = issue_dates[i]
    dates = pd.date_range(base, periods=HORIZONTE_MULTI)
    ax.plot(dates, obs_m[i], color=c, lw=1.5, marker="o", ms=4)
    ax.plot(dates, pred_m[i], color=c, lw=1.2, ls="--", marker="x", ms=5)

ax.set_ylabel("Q (m³/s)")
ax.set_title(
    f"MIMO h={HORIZONTE_MULTI}d — {n_show} trayectorias del test (sólido=obs, discontinua=pred)"
)
plt.tight_layout()

## 8 · Ejercicios

1. **GRU vs LSTM.** Cambia `LSTM` por `GRU(32)`. Compara tiempo de entrenamiento y RMSE.
2. **Ventana más larga.** Prueba `VENTANA=120` y `VENTANA=365`. ¿Mejora la captura de la estacionalidad?
3. **Multivariate con calendario.** Añade `sin_an`/`cos_an` como features. ¿Mejora?
4. **Stack LSTM.** `LSTM(32, return_sequences=True)` + `LSTM(16)`. Cuidado con overfit.
5. **Sparse MIMO.** Predice **sólo** $h \in \{7, 30\}$ (no los pasos intermedios). Generaliza `crear_ventanas_multistep` para aceptar una lista de horizontes:

   ```python
   HORIZONTES = [7, 30]

   def crear_ventanas_horizontes(df, features, target, ventana, horizontes):
       offsets = np.asarray(horizontes) - 1
       max_h = max(horizontes)
       X, Y, idx = [], [], []
       for i in range(len(df) - ventana - max_h + 1):
           X.append(df[features].values[i : i + ventana])
           Y.append(df[target].values[i + ventana + offsets])
           idx.append(df.index[i + ventana])
       return np.asarray(X), np.asarray(Y), pd.DatetimeIndex(idx)
   ```

   Entrena un modelo con `Dense(len(HORIZONTES))` y compara el RMSE en h=7 contra el MIMO denso de la Sección 7 (que predijo h=1..7). ¿Empeora, mejora o iguala?

   **Pitfall:** $y_{t+30}$ tiene mayor varianza que $y_{t+7}$. Considera normalizar **por columna** de `Y` (un `StandardScaler` distinto por horizonte) para que la pérdida no se dispare por h=30.